# 09 — Assemble the final report

**Foundry feature:** none new — this notebook rolls up everything notebooks 02-08 produced into the
one thing all of it was for: **is this candidate safe to promote?** **Mode: CODE.**

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## The promotion checklist

A candidate is promotable only when all four hold (see
`../prompt-agent-optimizer-baselines/docs/agent-evaluation-guide.md#what-counts-as-a-promotable-candidate`):

1. `validate_candidate.py` reports no critical failures (`blocked: false`) — notebook 05.
2. Every semantic rule left in the `UNJUDGED` queue has been resolved, by a real judge or a human,
   and none resolved to a critical failure — notebook 06.
3. Every gating test (`regression_blocks: true`) passes on **both** the optimize and holdout splits
   — a pass on optimize alone isn't sufficient evidence — notebook 07.
4. Instruction growth ratio and cost growth ratio stay under the agent's stated
   `max_instruction_growth_ratio` / `max_cost_growth_ratio` — a score win that costs a large,
   undisclosed token/latency increase isn't a clean win.

This cell checks what it can from the files this series has produced so far, and says plainly what's
still missing rather than guessing.

In [ ]:
expectations = json.loads((AGENT_DIR / "expected" / "expectations.json").read_text())
candidate_report_path = AGENT_DIR / "candidates" / "foundry_run1.report.json"

checklist = {}

if candidate_report_path.exists():
    report = json.loads(candidate_report_path.read_text())
    checklist["1. no critical failures (blocked=false)"] = (not report["blocked"], f"blocked={report['blocked']}")
    checklist["2. UNJUDGED queue resolved"] = (
        len(report.get("unjudged", [])) == 0,
        f"{len(report.get('unjudged', []))} item(s) still UNJUDGED -- run notebook 06 with a real judge backend",
    )
else:
    checklist["1. no critical failures (blocked=false)"] = (False, "no report found -- run notebook 05/06 first")
    checklist["2. UNJUDGED queue resolved"] = (False, "no report found -- run notebook 05/06 first")

checklist["3. gating tests pass on optimize AND holdout"] = (
    False, "requires a response-level harness against the deployed candidate -- see notebook 07's known gap",
)

scoring = expectations["scoring"]
checklist["4. growth ratios under cap"] = (
    False,
    f"needs instruction_token_count / est_cost from a filled-in run manifest "
    f"(caps: instructions <= {scoring['max_instruction_growth_ratio']}x, cost <= {scoring['max_cost_growth_ratio']}x)",
)

for item, (ok, detail) in checklist.items():
    print(f"[{'x' if ok else ' '}] {item}")
    print(f"      {detail}")

## What "safe to deploy" looks like specifically for this case study

`01-travel-approval-strict` is a **control** agent — the target outcome for this specific case study
is a *small* change, not a large one:

- All five `must_have` rules (the four approval-tier thresholds, self-approval limit, lodging caps
  and the 6-hour business-class rule, the restricted-destination hard stop, anti-fabrication) must
  survive the rewrite intact — verify this by eye against `candidates/foundry_run1.md`, not just by
  score.
- Expect a composite-score delta **under 0.03** — the pack's documented noise band for this agent. A
  large jump here is a signal to inspect the candidate closely, not a result to celebrate.
- The one legitimate fix is `should_remove:SR-01`, the hedged sign-off line — confirm it's gone and
  nothing else changed materially.

In [ ]:
baseline_text = (AGENT_DIR / "instructions.md").read_text()
candidate_path = AGENT_DIR / "candidates" / "foundry_run1.md"
if candidate_path.exists():
    candidate_text = candidate_path.read_text()
    print(f"baseline instructions : {len(baseline_text.split())} words")
    print(f"candidate instructions: {len(candidate_text.split())} words")
    print(f"growth ratio (word-count proxy): {len(candidate_text.split()) / len(baseline_text.split()):.2f}x "
          f"(cap: {expectations['scoring']['max_instruction_growth_ratio']}x)")

## Summary table shape for a written report

Once notebooks 05-08 have real (not stand-in) data behind them across enough replicate runs, this is
the table every number in a write-up should trace back to:

| Column | Source |
|---|---|
| Baseline vs. optimized composite score, per track | `compare_to_foundry.py` output — notebook 08 |
| Regression-gate pass rate | `validate_candidate.py` / `run_manifest.json`'s critical-failure field, per replicate seed — notebooks 05, 07 |
| Primary/cross-judge agreement | `primary_cross_judge_agreement` in each judged report — notebook 06 |
| Cross-run textual similarity | `similarity_baseline.py` output — notebook 08 |
| Instruction and cost growth ratio | `run_manifest.json`'s growth-ratio fields — this notebook |

Keep every raw `run_manifest.json`, candidate file, and validation report this series produced —
a report should cite these files directly rather than a number transcribed by hand, so any figure can
be traced back to the exact run that produced it.

## Re-running this series against a different case study

Everything above generalizes to any of the pack's other nine agents — go back to notebook 00, change
`AGENT_ID`, and re-run. A few things worth trying deliberately different:

- **`02-support-triage-messy`** — a genuinely poor baseline (a buried privacy violation) instead of a
  control, to see the optimizer asked to fix something real.
- **`04-hr-policy-mcp`** or **`07-incident-response-mcp`** — an MCP-backed agent, where retrieval
  behavior has to live entirely in the instructions because the optimizer can't touch MCP tool
  descriptions.
- **`05-clinical-triage-safety`** — the highest-stakes agent in the pack, where the interesting flaw
  is two quietly-unsafe lines at the end of an otherwise well-written prompt.

See `../prompt-agent-optimizer-baselines/docs/agent-evaluation-guide.md` for what every agent is
designed to expose before you pick one.